# पाठ 03 - एजेंटिक डिज़ाइन पैटर्न

इस पाठ में, हम प्रभावी AI एजेंट बनाने के लिए तीन मूलभूत डिज़ाइन पैटर्न का अन्वेषण करेंगे:

1. **स्पष्ट एजेंट निर्देश** — एजेंट व्यवहार को मार्गदर्शित करने वाले सटीक, भूमिका-निर्धारित प्रॉम्प्ट तैयार करना  
2. **पाइडैंटिक मॉडल के साथ संरचित आउटपुट** — यह सुनिश्चित करना कि एजेंट पूर्वानुमानित, सत्यापित डेटा लौटाएँ  
3. **सिंगल रिस्पॉन्सिबिलिटी एजेंट्स** — ऐसे केंद्रित एजेंट डिजाइन करना जो प्रत्येक केवल एक काम कुशलता से करें  

हम प्रत्येक पैटर्न को एक **यात्रा गंतव्य अनुशंसक** परिदृश्य में लागू करेंगे, क्रमशः एक ऐसा सिस्टम बनाएंगे जो गंतव्य सुझा सके, उपलब्धता जांच सके, और लॉजिस्टिक्स संभाल सके।


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## पैटर्न 1: स्पष्ट एजेंट निर्देश

सबसे प्रभावशाली पैटर्न सबसे सरल भी है: अपने एजेंट के लिए स्पष्ट, विस्तार से निर्देश लिखना।

अच्छे निर्देश परिभाषित करते हैं:
- **कौन** एजेंट है (व्यक्तित्व और अंदाज)
- **क्या** उसे करना चाहिए (चरण-दर-चरण जिम्मेदारियां)
- **कैसे** उसे व्यवहार करना चाहिए (सीमाएं और शैली)

नीचे, हम एक यात्रा कंसीयर्ज़ एजेंट बनाते हैं जिसमें स्पष्ट निर्देश होते हैं जो इसके द्वारा दी जाने वाली हर प्रतिक्रिया को आकार देते हैं।


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## पैटर्न 2: पायडेंटिक मॉडल के साथ संरचित आउटपुट

फ्री-फॉर्म टेक्स्ट बातचीत के लिए उपयोगी है, लेकिन डाउनस्ट्रीम सिस्टम को संरचित डेटा की आवश्यकता होती है।  
**पायडेंटिक मॉडल** को एक **टूल फ़ंक्शन** के साथ जोड़कर, हम:

- एजेंट के आउटपुट के लिए एक सटीक स्कीमा परिभाषित कर सकते हैं  
- प्रतिक्रियाओं को स्वचालित रूप से वैध कर सकते हैं  
- एजेंट के परिणामों को विश्वसनीय रूप से एप्लिकेशन लॉजिक में एकीकृत कर सकते हैं  

हम एक ऐसा टूल भी पेश करते हैं जो गंतव्य विवरण लौटाता है ताकि एजेंट अपनी सिफारिशों को वास्तविक डेटा में आधारित कर सके।


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## पैटर्न 3: एकल जिम्मेदारी एजेंट

जटिल कार्यों को कई केंद्रित एजेंट्स में विभाजित करने से लाभ मिलता है, जिनमें से प्रत्येक की एक एकल जिम्मेदारी होती है:

- एक **डेस्टिनेशन एक्सपर्ट** जो स्थानों और उपलब्धता के बारे में जानता है
- एक **लॉजिस्टिक्स प्लानर** जो उड़ानों, होटलों और यात्रा कार्यक्रमों को संभालता है

यह सॉफ्टवेयर इंजीनियरिंग सिद्धांत *सेपरेशन ऑफ कॉन्सर्न्स* को दर्शाता है — प्रत्येक एजेंट को स्वतंत्र रूप से परीक्षण, रखरखाव और सुधार करना आसान होता है।


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## सारांश

इस पाठ में हमने एक यात्रा सिफारिशकर्ता परिदृश्य पर तीन एजेंटिक डिज़ाइन पैटर्न लागू किए:

| पैटर्न | मुख्य विचार | लाभ |
|---|---|---|
| **स्पष्ट निर्देश** | प्रारंभ में ही व्यक्ति, जिम्मेदारियां, और सीमाएं परिभाषित करें | लगातार, ब्रांड के अनुरूप एजेंट व्यवहार |
| **संरचित आउटपुट** | प्रतिक्रिया स्वरूप के रूप में Pydantic मॉडल का उपयोग करें | मान्य, मशीन-पठनीय परिणाम |
| **एकल ज़िम्मेदारी** | प्रत्येक एजेंट को एक केंद्रित कार्य दें | परीक्षण, रखरखाव, और संयोजन में आसान |

ये पैटर्न स्वाभाविक रूप से मेल खाते हैं — आप स्पष्ट निर्देशों को संरचित आउटपुट के साथ एकल-जिम्मेदारी एजेंट के अंदर जोड़ सकते हैं ताकि मज़बूत, उत्पादन-तैयार प्रणालियाँ बनायीं जा सकें।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यह दस्तावेज़ एआई अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके अनुवादित किया गया है। जबकि हम सटीकता के लिए प्रयासरत हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या असंगतियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मातृ भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
